# 복지정책 데이터 전처리 (Bronze → Silver Markdown 변환)

## 개요
크롤러로 수집한 원본 JSON(중앙정부/지자체/민간/청년정책)을 RAG 파이프라인이 바로 사용할 수 있는 형태로 가공하는 전처리 코드입니다. 각 소스별로 필드 구성이 달라서(예: 지자체는 지자체명·표 정보, 민간은 사업상태·연락처 등) 소스마다 별도의 변환 함수를 두고, 정책 1건을 Markdown 텍스트로 만든 뒤 `doc_id` / `source_type` / `content` 등을 포함한 Silver JSON 문서로 저장합니다. 이렇게 만들어진 Silver JSON이 다중쿼리·단일쿼리 노트북에서 `Document`로 로드되는 원천 데이터입니다.

## 주요 기능
1. **소스별 Markdown 변환** — 중앙정부/지자체/민간/청년정책 각각 다른 필드 구조를 공통된 Markdown 형식(`# 제목`, `- 담당부처`, `[서비스내용]` 등)으로 통일합니다.
2. **Silver 문서 스키마 변환** — 변환된 Markdown을 `doc_id`, `source_type`, `source_name`, `content`, `created_at`, `metadata`를 가진 표준 문서 형식으로 감쌉니다.
3. **수동 문서 생성 유틸** — 크롤링 없이 직접 작성한 Markdown 텍스트 한 건을 Silver JSON으로 변환하는 범용 함수도 별도로 제공합니다.

## 코드 구조
- **1~2번**: 중앙정부 JSON → Markdown 변환 → Silver JSON 저장
- **3번**: 지자체 JSON → Markdown 변환 → Silver JSON 저장
- **4번**: 민간 JSON → Markdown 변환 → Silver JSON 저장
- **5번**: 단일 Markdown 텍스트 → Silver JSON 변환 (범용 유틸)
- **6번**: 청년정책 JSON → Markdown 변환 → Silver JSON 저장

## 설계 포인트
- **소스별 스키마 분리**: 하나의 변환 함수로 억지로 통합하지 않고, 소스마다 실제 원본 필드 구조가 달라서 `policy_json_to_markdown` 함수를 소스별로 따로 정의했습니다.
- **doc_id 규칙화**: `generate_doc_id`로 `{source_type}_{source_name}_{순번}` 형태의 고정 규칙을 적용해, 이후 RAG 파이프라인에서 문서 출처를 추적할 수 있게 했습니다.
- **메타데이터 최소화**: `metadata`에는 검색/필터링에 바로 쓸 수 있는 키워드·서비스내용 정도만 남기고, 본문은 전부 `content`(Markdown)에 담아 LLM이 그대로 읽을 수 있게 구성했습니다.


**1. 중앙정부 정책 → 마크다운 변환**


In [ ]:
#중앙정부
from typing import Dict, List
from datetime import datetime
import json
from pathlib import Path

# ------------------------------
# 정책 dict → 마크다운 변환 함수
# ------------------------------
def policy_json_to_markdown(policy: Dict) -> str:
    """
    정책 1건(dict)을 Markdown 텍스트로 변환.
    sections 안의 모든 키도 [키] 형태로 포함.
    개정일(lastMdfcnDt)도 표시.
    마지막에 [복지사례] 항목도 항상 추가.
    """
    title = policy.get("제목", "제목 없음")
    구분 = policy.get("구분", "")
    배지 = policy.get("배지", "")
    서비스내용 = policy.get("서비스내용", "")
    담당부처 = policy.get("담당부처", "")
    표_기준연도 = policy.get("표_기준연도", "")
    표_문의처 = policy.get("표_문의처", "")
    표_지원주기 = policy.get("표_지원주기", "")
    표_제공유형 = policy.get("표_제공유형", "")
    last_modified = policy.get("lastMdfcnDt", "")
    sections = policy.get("sections", {})

    lines = []

    # 제목(H1)
    lines.append(f"# {title}")
    lines.append("")

    # 기본 정보

    if 구분:
        lines.append(f"- 구분: {구분}")
    if 배지:
        lines.append(f"- 배지: {배지}")
    if 담당부처:
        lines.append(f"- 담당부처: {담당부처}")
    if 표_기준연도:
        lines.append(f"- 표_기준연도: {표_기준연도}")
    if 표_문의처:
        lines.append(f"- 표_문의처: {표_문의처}")
    if 표_지원주기:
        lines.append(f"- 표_지원주기: {표_지원주기}")
    if 표_제공유형:
        lines.append(f"- 표_제공유형: {표_제공유형}")
    if last_modified:
        lines.append(f"- 마지막 개정일: {last_modified}")
    lines.append("")

    # 서비스 내용
    if 서비스내용:
        lines.append(f"[서비스내용]\n{서비스내용}")
        lines.append("")

    # sections 안의 모든 항목 Markdown으로 추가 (값이 있으면)
    for key, value in sections.items():
        if key != "복지사례" and value not in (None, "", []):
            lines.append(f"[{key}]\n{value}")
            lines.append("")

    # 항상 [복지사례] 항목 추가 (없으면 빈 문자열)
    lines.append(f"[복지사례]\n{sections.get('복지사례', '')}")
    lines.append("")

    # 모든 줄 합치기
    return "\n".join(lines)


# ------------------------------
# doc_id 생성 함수
# ------------------------------
def generate_doc_id(source_type: str, source_name: str, index: int) -> str:
    return f"{source_type}_{source_name}_{index:04d}"


# ------------------------------
# JSON → Silver 문서 리스트 변환
# ------------------------------
def process_youth_policy_json(filepath: Path) -> List[Dict]:
    """
    JSON 파일을 읽어서 Silver 형식의 문서 리스트로 변환.
    content는 Markdown 형태로 들어감.
    """
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON은 리스트[dict, dict, ...] 구조여야 합니다.")

    documents: List[Dict] = []

    stem = filepath.stem
    if "_" in stem:
        prefix, rest = stem.split("_", 1)
        source_type = prefix
        source_name = rest
    else:
        source_type = "정책"
        source_name = stem

    source_file = filepath.name
    today_str = datetime.now().strftime("%Y-%m-%d")

    for i, policy in enumerate(data, start=1):
        content_md = policy_json_to_markdown(policy)
        doc_id = generate_doc_id(source_type, source_name, i)

        doc = {
            "doc_id": doc_id,
            "source_type": source_type,
            "source_name": source_name,
            "source_file": source_file,
            "content": content_md,
            "created_at": today_str,
            "metadata": {
                "키워드": policy.get("배지", ""),
                "서비스내용": policy.get("서비스내용", "")
            }
        }
        documents.append(doc)

    return documents


**2. Silver JSON 저장 및 실행 (중앙정부)**


In [ ]:
# ------------------------------
# Silver 문서 JSON 저장 함수
# ------------------------------
def save_silver_json(documents: List[Dict], output_dir: str, output_filename: str) -> Path:
    """
    Silver 문서 리스트를 JSON 파일로 저장
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    file_path = output_path / output_filename

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)

    print(f"✅ 저장 완료: {file_path}")
    print(f"📊 총 {len(documents)}개 문서 처리됨")
    return file_path


# ------------------------------
# 사용 예시
# ------------------------------
if __name__ == "__main__":
    input_file = Path("/content/복지서비스_서비스목록_중앙부처.json")
    output_dir = "./silver_data"
    output_filename = "복지서비스_서비스목록_중앙부처.json"

    documents = process_youth_policy_json(input_file)
    save_silver_json(documents, output_dir, output_filename)


✅ 저장 완료: silver_data/복지서비스_서비스목록_중앙부처.json
📊 총 95개 문서 처리됨


**3. 지자체 정책 → 마크다운 변환 및 저장**


In [ ]:
from typing import Dict, List
from datetime import datetime
import json
from pathlib import Path

# ------------------------------
# 정책 dict → 마크다운 변환 함수
# ------------------------------
def policy_json_to_markdown(policy: Dict) -> str:
    """
    정책 1건(dict)을 Markdown 텍스트로 변환.
    sections 안의 모든 키도 [키] 형태로 포함.
    마지막에 [복지사례] 항목도 항상 추가.
    """
    title = policy.get("basic_info", {}).get("제목", "제목 없음")
    페이지 = policy.get("페이지", "")
    항목번호 = policy.get("항목번호", "")
    배지 = policy.get("basic_info", {}).get("배지", "")
    서비스내용 = policy.get("basic_info", {}).get("서비스소개", "")
    담당부처 = policy.get("basic_info", {}).get("담당부처", "")
    지원주기 = policy.get("basic_info", {}).get("지원주기", "")
    신청방법 = policy.get("basic_info", {}).get("신청방법", "")
    제공유형 = policy.get("basic_info", {}).get("제공유형", "")
    sections = policy.get("sections", {})

    lines = []

    # 제목(H1)
    lines.append(f"# {title}")
    lines.append("")

    # 기본 정보
    if 배지:
        lines.append(f"- 배지: {배지}")
    if 담당부처:
        lines.append(f"- 담당부처: {담당부처}")
    if 지원주기:
        lines.append(f"- 지원주기: {지원주기}")
    if 신청방법:
        lines.append(f"- 신청방법: {신청방법}")
    if 제공유형:
        lines.append(f"- 제공유형: {제공유형}")
    lines.append("")

    # 서비스 내용
    if 서비스내용:
        lines.append(f"[서비스내용]\n{서비스내용}")
        lines.append("")

    # sections 안의 모든 항목 Markdown으로 추가 (값이 있으면)
    for key, value in sections.items():
        if key != "복지사례" and value not in (None, "", []):
            lines.append(f"[{key}]\n{value}")
            lines.append("")

    # 항상 [복지사례] 항목 추가 (없으면 빈 문자열)
    lines.append(f"[복지사례]\n{sections.get('복지사례', '')}")
    lines.append("")

    return "\n".join(lines)

# ------------------------------
# doc_id 생성 함수
# ------------------------------
def generate_doc_id(source_type: str, source_name: str, index: int) -> str:
    return f"{source_type}_{source_name}_{index:04d}"

# ------------------------------
# JSON → Silver 문서 리스트 변환
# ------------------------------
def process_local_gov_policy_json(filepath: Path) -> List[Dict]:
    """
    지자체 JSON 파일을 읽어서 Silver 형식 문서 리스트로 변환.
    content는 Markdown 형태.
    """
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON은 리스트[dict, dict, ...] 구조여야 합니다.")

    documents: List[Dict] = []

    # 고정 source_type / source_name
    source_type = "지자체"
    source_name = "복지서비스 목록(지자체)"
    source_file = filepath.name
    today_str = datetime.now().strftime("%Y-%m-%d")

    for i, policy in enumerate(data, start=1):
        content_md = policy_json_to_markdown(policy)
        doc_id = generate_doc_id(source_type, source_name, i)

        doc = {
            "doc_id": doc_id,
            "source_type": source_type,
            "source_name": source_name,
            "source_file": source_file,
            "content": content_md,
            "created_at": today_str,
            "metadata": {
                "키워드": policy.get("basic_info", {}).get("배지", ""),
                "서비스내용": policy.get("basic_info", {}).get("서비스소개", "")
            }
        }
        documents.append(doc)

    return documents

# ------------------------------
# Silver JSON 파일로 저장
# ------------------------------
def save_documents_to_json(documents: List[Dict], output_path: str):
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)
    print(f"✅ 저장 완료: {output_file}")
    print(f"📄 총 {len(documents)}개 문서 처리됨")

# ------------------------------
# 사용 예시
# ------------------------------
if __name__ == "__main__":
    input_file = Path("/content/복지서비스_서비스 목록_지자체.json")  # 원본 JSON 파일

    # 저장 경로 수
    output_file = Path("/content/silver_복지서비스_서비스목록_지자체.json")

    docs = process_local_gov_policy_json(input_file)
    save_documents_to_json(docs, output_file)



✅ 저장 완료: /content/silver_복지서비스_서비스목록_지자체.json
📄 총 1205개 문서 처리됨


**4. 민간 정책 → 마크다운 변환 및 저장**


In [ ]:
#민간
from typing import Dict, List
from datetime import datetime
import json
from pathlib import Path

# ------------------------------
# 정책 dict → 마크다운 변환 함수 (민간용)
# ------------------------------
def policy_json_to_markdown(policy: Dict) -> str:
    """
    정책 1건(dict)을 Markdown 텍스트로 변환.
    sections 안의 모든 키도 [키] 형태로 포함.
    마지막에 [복지사례] 항목도 항상 추가.
    """
    title = policy.get("basic_info", {}).get("제목", "제목 없음")
    페이지 = policy.get("페이지", "")
    항목번호 = policy.get("항목번호", "")
    배지 = policy.get("basic_info", {}).get("배지", "")
    서비스내용 = policy.get("basic_info", {}).get("서비스소개", "")
    담당부처 = policy.get("basic_info", {}).get("담당부처", "")
    사업상태 = policy.get("basic_info", {}).get("사업상태", "")
    사업기간 = policy.get("basic_info", {}).get("사업기간", "")
    연락처 = policy.get("basic_info", {}).get("연락처", "")
    이메일 = policy.get("basic_info", {}).get("이메일", "")
    sections = policy.get("sections", {})

    lines = []

    # 제목(H1)
    lines.append(f"# {title}")
    lines.append("")

    # 기본 정보
    if 페이지:
        lines.append(f"- 페이지: {페이지}")
    if 항목번호:
        lines.append(f"- 항목번호: {항목번호}")
    if 배지:
        lines.append(f"- 배지: {배지}")
    if 담당부처:
        lines.append(f"- 담당부처: {담당부처}")
    if 사업상태:
        lines.append(f"- 사업상태: {사업상태}")
    if 사업기간:
        lines.append(f"- 사업기간: {사업기간}")
    if 연락처:
        lines.append(f"- 연락처: {연락처}")
    if 이메일:
        lines.append(f"- 이메일: {이메일}")
    if 서비스내용:
        lines.append(f"- 서비스소개: {서비스내용}")
    lines.append("")

    # sections 안의 모든 항목 Markdown으로 추가 (값이 있으면)
    for key, value in sections.items():
        if key != "복지사례" and value not in (None, "", []):
            lines.append(f"[{key}]\n{value}")
            lines.append("")
  # 항상 [복지사례] 항목 추가 (없으면 빈 문자열)
    lines.append(f"[복지사례]\n{sections.get('복지사례', '')}")
    lines.append("")

    return "\n".join(lines)


# ------------------------------
# doc_id 생성 함수
# ------------------------------
def generate_doc_id(source_type: str, source_name: str, index: int) -> str:
    return f"{source_type}_{source_name}_{index:04d}"

# ------------------------------
# JSON → Silver 문서 리스트 변환 (민간용)
# ------------------------------
def process_private_policy_json(filepath: Path) -> List[Dict]:
    """
    민간 JSON 파일을 읽어서 Silver 형식 문서 리스트로 변환.
    content는 Markdown 형태.
    """
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON은 리스트[dict, dict, ...] 구조여야 합니다.")

    documents: List[Dict] = []

    # 고정 source_type / source_name
    source_type = "민간"
    source_name = "복지서비스 목록(민간)"
    source_file = filepath.name
    today_str = datetime.now().strftime("%Y-%m-%d")

    for i, policy in enumerate(data, start=1):
        content_md = policy_json_to_markdown(policy)
        doc_id = generate_doc_id(source_type, source_name, i)

        doc = {
            "doc_id": doc_id,
            "source_type": source_type,
            "source_name": source_name,
            "source_file": source_file,
            "content": content_md,
            "created_at": today_str,
            "metadata": {
                "키워드": policy.get("basic_info", {}).get("배지", ""),
                "서비스내용": policy.get("basic_info", {}).get("서비스소개", "")
            }
        }
        documents.append(doc)

    return documents

# ------------------------------
# Silver JSON 파일로 저장
# ------------------------------
def save_documents_to_json(documents: List[Dict], output_path: str):
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)
    print(f"✅ 저장 완료: {output_file}")
    print(f"📄 총 {len(documents)}개 문서 처리됨")

# ------------------------------
# 사용 예시
# ------------------------------
if __name__ == "__main__":
    input_file = Path("/content/복지서비스_서비스 목록_민간.json")  # 민간 원본 JSON 파일
    output_file = Path("./silver_data/silver_복지서비스_서비스목록_민간.json")

    docs = process_private_policy_json(input_file)
    save_documents_to_json(docs, output_file)


**5. 단일 마크다운 → Silver JSON 변환 (수동 입력용)**


In [ ]:
from datetime import datetime
from pathlib import Path
import json

# ------------------------------
# 단일 마크다운 → Silver JSON 변환
# ------------------------------
def create_markdown_silver_doc(md_content: str, doc_id: str, source_type: str, source_name: str, source_file: str) -> dict:
    """
    마크다운 텍스트를 Silver JSON 문서 형식으로 변환
    metadata는 빈 dict로 처리
    """
    today_str = datetime.now().strftime("%Y-%m-%d")

    doc = {
        "doc_id": doc_id,
        "source_type": source_type,
        "source_name": source_name,
        "source_file": source_file,
        "content": md_content,
        "created_at": today_str,
        "metadata": {}
    }

    return doc

# ------------------------------
# JSON 파일로 저장
# ------------------------------
def save_single_document(doc: dict, output_path: str):
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)
    print(f"✅ 저장 완료: {output_file}")

# ------------------------------
# 사용 예시
# ------------------------------
# markdown_content에 변환할 텍스트를 입력하고 아래 함수들을 실행하세요.
# doc = create_markdown_silver_doc(markdown_content, doc_id, source_type, source_name, source_file)
# save_single_document(doc, output_path)


✅ 저장 완료: silver_data/silver_맞춤형급여안내.json


**6. 청년정책 → 마크다운 변환 및 저장**


In [17]:
from typing import Dict, List
from datetime import datetime
import json
from pathlib import Path

# ------------------------------
# 정책 dict → 마크다운 변환 함수
# ------------------------------
def policy_json_to_markdown(policy: Dict) -> str:
    """
    청년정책 1건(dict)을 Markdown 텍스트로 변환.
    sections 대신 아래 9개 키를 포함.
    """

    title = policy.get("제목", "제목 없음")
    subtitle = policy.get("부제", "")
    description = policy.get("설명", "")
    provider = policy.get("제공기관", "")
    keywords = policy.get("키워드", "")
    url = policy.get("정보URL", "")
    last_modified = policy.get("최종수정일", "")

    lines = []

    # 제목(H1)
    lines.append(f"# {title}")
    lines.append("")

    # 기본 정보

    if subtitle:
        lines.append(f"- 부제: {subtitle}")
    if provider:
        lines.append(f"- 제공기관: {provider}")
    if keywords:
        lines.append(f"- 키워드: {keywords}")
    if url:
        lines.append(f"- 정보URL: {url}")

    if last_modified:
        lines.append(f"- 최종수정일: {last_modified}")
    lines.append("")

    # 설명
    if description:
        lines.append(f"[설명]\n{description}")
        lines.append("")

    return "\n".join(lines)

# ------------------------------
# doc_id 생성 함수
# ------------------------------
def generate_doc_id(source_type: str, source_name: str, index: int) -> str:
    # 고정 doc_id
     return f"{source_type}_{source_name}_{index:04d}"

# ------------------------------
# JSON → Silver 문서 리스트 변환
# ------------------------------
def process_local_gov_policy_json(filepath: Path) -> List[Dict]:
    """
    청년정책 JSON 파일을 읽어서 Silver 형식 문서 리스트로 변환.
    content는 Markdown 형태.
    """
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON은 리스트[dict, dict, ...] 구조여야 합니다.")

    documents: List[Dict] = []

    # 고정 source_type / source_name / source_file
    source_type = "청년정책"
    source_name = "청년정책 바로가기"
    source_file = "청년정책_바로가기.txt"
    today_str = datetime.now().strftime("%Y-%m-%d")

    for i, policy in enumerate(data, start=1):
        content_md = policy_json_to_markdown(policy)
        doc_id = generate_doc_id(source_type, source_name, i)

        doc = {
            "doc_id": doc_id,
            "source_type": source_type,
            "source_name": source_name,
            "source_file": source_file,
            "content": content_md,
            "created_at": today_str,
            "metadata": {
                "키워드": policy.get("키워드", ""),
                "설명": policy.get("설명", "")
            }
        }
        documents.append(doc)

    return documents

# ------------------------------
# Silver JSON 파일로 저장
# ------------------------------
def save_documents_to_json(documents: List[Dict], output_path: str):
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)
    print(f"✅ 저장 완료: {output_file}")
    print(f"📄 총 {len(documents)}개 문서 처리됨")

# ------------------------------
# 사용 예시
# ------------------------------
if __name__ == "__main__":
    input_file = Path("/content/청년정책_바로가기.json")  # 청년정책 원본 JSON 파일
    output_file = Path("./silver_data/silver_청년정책_바로가기.json")

    docs = process_local_gov_policy_json(input_file)
    save_documents_to_json(docs, output_file)


✅ 저장 완료: silver_data/silver_청년정책_바로가기.json
📄 총 74개 문서 처리됨
